<a href="https://colab.research.google.com/github/tamkj/CapStoneETKJ/blob/main/CapStone_MLearng.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Predicts whether an Olympic athlete–event entry won a medal (won_medal: True/False)
# using metadata features (Age, Height, Weight, BMI, Sex, SportKey, NOCKey).
# Model: Logistic Regression (class-weighted for class imbalance) in an 80/20 stratified split.


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# ---------- 1. Loan table ------------------------
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/1.NTUC/11.CapStone/FactParticipation.csv')

# ---------- 2. Select features & target ----------
target = "won_medal"

# Keep simple, informative columns
num_features = ["Age", "Height", "Weight", "BMI"]
cat_features = ["Sex", "SportKey", "NOCKey"]

X = df[num_features + cat_features].copy()
y = df[target].astype(int)

# Optional: reduce cardinality of NOCKey / SportKey (keep top N, rest -> "Other")
def reduce_cardinality(series, top_n):
    top = series.value_counts().nlargest(top_n).index
    return series.where(series.isin(top), "Other")

X["NOCKey"] = reduce_cardinality(X["NOCKey"], 30)
X["SportKey"] = reduce_cardinality(X["SportKey"], 20)

# ---------- 3. Preprocessing pipeline ----------
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_features),
    ("cat", categorical_pipe, cat_features),
])

# ---------- 4. Model pipeline ----------
model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",   # handles rare medal class
        solver="lbfgs",
    )),
])

# ---------- 5. Train / test split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------- 6. Fit & evaluate ----------
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROC-AUC: 0.7212
